In [ ]:
# lab4_vector_clock.py
class VectorClock:
    def __init__(self, pid, all_pids):
        self.pid = pid
        self.clock = {p: 0 for p in all_pids}

    def tick(self):
        self.clock[self.pid] += 1
        return dict(self.clock)

    def send(self):
        self.clock[self.pid] += 1
        return dict(self.clock)

    def receive(self, remote_ts):
        for p in self.clock:
            self.clock[p] = max(self.clock[p], remote_ts[p])
        self.clock[self.pid] += 1  # increment setelah merge

    def happens_before(self, ts_a, ts_b) -> bool:
        """Return True jika ts_a causally precedes ts_b."""
        leq = all(ts_a[p] <= ts_b[p] for p in ts_a)
        lt = any(ts_a[p] < ts_b[p] for p in ts_a)
        return leq and lt

    def concurrent(self, ts_a, ts_b) -> bool:
        return (not self.happens_before(ts_a, ts_b) and
                not self.happens_before(ts_b, ts_a))

# Demo
pids = ["P1", "P2", "P3"]
vc1 = VectorClock("P1", pids)
vc2 = VectorClock("P2", pids)
vc3 = VectorClock("P3", pids)

ts_a = vc1.send()  # P1 sends
vc2.receive(ts_a)
ts_b = vc2.send()  # P2 sends after receiving

print("ts_a:", ts_a)
print("ts_b:", ts_b)
print("a → b?", vc1.happens_before(ts_a, ts_b))  # True
print("b → a?", vc1.happens_before(ts_b, ts_a))  # False
print("concurrent?", vc1.concurrent(ts_a, ts_b))  # False (a→b, bukan concurrent)

# Skenario concurrent: P1 dan P3 kirim pesan secara bersamaan ke P2
vc3 = VectorClock("P3", pids)
ts_c = vc3.send()  # P3 kirim tanpa tahu tentang ts_a
print("\nts_a (dari P1):", ts_a)
print("ts_c (dari P3):", ts_c)
print("a → c?", vc1.happens_before(ts_a, ts_c))  # False
print("c → a?", vc1.happens_before(ts_c, ts_a))  # False
print("a || c (concurrent)?", vc1.concurrent(ts_a, ts_c))  # True!

ts_a: {'P1': 1, 'P2': 0, 'P3': 0}
ts_b: {'P1': 1, 'P2': 2, 'P3': 0}
a → b? True
b → a? False
concurrent? False

ts_a (dari P1): {'P1': 1, 'P2': 0, 'P3': 0}
ts_c (dari P3): {'P1': 0, 'P2': 0, 'P3': 1}
a → c? False
c → a? False
a || c (concurrent)? True


In [9]:
class Node:
    def __init__(self, node_id, total_nodes):
        self.node_id = node_id
        self.vector_clock = [0] * total_nodes

    def send_message(self, receiver, message_text):
        self.vector_clock[self.node_id] += 1
        message = Message(self.node_id, message_text, list(self.vector_clock))
        receiver.receive_message(message)
# n1 : [1, 0, 0]
# n2 : [1, 2, 0]
    def receive_message(self, message):
        self.vector_clock = [max(vc1, vc2) for vc1, vc2 in zip(self.vector_clock, message.vector_clock)]
        self.vector_clock[self.node_id] += 1
        print(f"Node {self.node_id} received message: {message.text}")
        print(f"Updated vector clock : {self.vector_clock}")
# n2 : [1, 1, 0]
# n3 : [0, 0, 1]
    def __str__(self):
        return f"Node {self.node_id} - Vector Clock: {self.vector_clock}"

class Message:
    def __init__(self, sender_id, text, vector_clock):
        self.sender_id = sender_id
        self.text = text
        self.vector_clock = vector_clock

class DistributedSystem:
    def __init__(self, num_nodes):
        self.nodes = [Node(i, num_nodes) for i in range(num_nodes)]

    def simulate(self):
        # Node 0 sends a message to Node 1
        self.nodes[0].send_message(self.nodes[1], "Hello from Node 0")

        # Node 1 sends a message to Node 2
        self.nodes[1].send_message(self.nodes[2], "Hello from Node 1")

        # Node 2 sends a message to Node 0
        self.nodes[2].send_message(self.nodes[0], "Hello from Node 2")

        # Print final vector clocks
        for node in self.nodes:
            print(node)

# Example simulation
system = DistributedSystem(3)
system.simulate()

Node 1 received message: Hello from Node 0
Updated vector clock : [1, 1, 0]
Node 2 received message: Hello from Node 1
Updated vector clock : [1, 2, 1]
Node 0 received message: Hello from Node 2
Updated vector clock : [2, 2, 2]
Node 0 - Vector Clock: [2, 2, 2]
Node 1 - Vector Clock: [1, 2, 0]
Node 2 - Vector Clock: [1, 2, 2]
